In [2]:
import pandas as pd
import numpy as np
import os

ea_dir = r"C:\Users\user\Downloads\GSE148812_clean"

meta_df = pd.read_csv(os.path.join(ea_dir, "checkpoint2b_metadata_relatedness_filtered.csv"))
meta_df["sample_id"] = meta_df["sample_id"].astype(str)
meta_df = meta_df.set_index("sample_id")
meta_df["smoking_status_bin"] = (meta_df["smoking_status"] == "Smoker").astype(int)

gene_burden_ea = pd.read_csv(os.path.join(ea_dir, "gene_burden_matrix_signed_protein_coding_EA.csv"), index_col=0)
sample_cols = gene_burden_ea.columns.tolist()

pheno_ea = meta_df.loc[sample_cols, "smoking_status_bin"].astype(float)
print("EA phenotype reloaded:", pheno_ea.shape)

EA phenotype reloaded: (1460,)


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from scipy import stats
import time
import os

ea_dir = r"C:\Users\user\Downloads\GSE148812_clean"

gene_burden_ea = pd.read_csv(os.path.join(ea_dir, "gene_burden_matrix_signed_protein_coding_EA.csv"), index_col=0)
gene_names_ea = gene_burden_ea.index.tolist()

X_genes_ea = gene_burden_ea.T.values
X_standardized_ea = (X_genes_ea - X_genes_ea.mean(axis=0)) / X_genes_ea.std(axis=0)

Y_ea = pheno_ea.values

X_confounders_ea = np.load(os.path.join(ea_dir, "confounders_X_relatedness_filtered.npy"))
print("EA confounders:", X_confounders_ea.shape)

def doubleml_scan(X_snps, Y, X_conf, n_folds=5, random_state=42):
    n, n_snps = X_snps.shape
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    D_resid = np.zeros_like(X_snps)
    Y_resid = np.zeros(n)
    Xc = np.column_stack([np.ones(n), X_conf])

    for train_idx, test_idx in kf.split(Xc):
        Xc_tr, Xc_te = Xc[train_idx], Xc[test_idx]
        coef_Y = np.linalg.lstsq(Xc_tr, Y[train_idx], rcond=None)[0]
        Y_resid[test_idx] = Y[test_idx] - Xc_te @ coef_Y
        coef_D = np.linalg.lstsq(Xc_tr, X_snps[train_idx], rcond=None)[0]
        D_resid[test_idx] = X_snps[test_idx] - Xc_te @ coef_D

    Yr = Y_resid - Y_resid.mean()
    Dr = D_resid - D_resid.mean(axis=0)
    del D_resid

    sum_DY = (Dr * Yr[:, None]).sum(axis=0)
    sum_DD = (Dr ** 2).sum(axis=0)
    sum_YY = (Yr ** 2).sum()

    theta = sum_DY / sum_DD
    ssr = sum_YY - (sum_DY ** 2) / sum_DD
    sigma2 = ssr / (n - 2)
    se = np.sqrt(sigma2 / sum_DD)
    t_stat = theta / se
    pvals = 2 * (1 - stats.t.cdf(np.abs(t_stat), df=n - 2))
    return theta, pvals

n_repeats = 30
threshold = 0.001
n_genes_ea = X_standardized_ea.shape[1]
significant_counts_ea = np.zeros(n_genes_ea, dtype=int)

start = time.time()
for rep in range(n_repeats):
    theta_rep, pval_rep = doubleml_scan(X_standardized_ea, Y_ea, X_confounders_ea, random_state=rep)
    significant_counts_ea += (pval_rep < threshold).astype(int)
    if (rep + 1) % 5 == 0:
        print(f"Completed {rep+1}/{n_repeats}, elapsed {time.time()-start:.1f}s")

print(f"Total time: {time.time()-start:.1f}s")

stability_fraction_ea = significant_counts_ea / n_repeats
results_df_ea = pd.DataFrame({
    "gene": gene_names_ea,
    "stability_fraction": stability_fraction_ea,
    "n_significant_repeats": significant_counts_ea
}).sort_values("stability_fraction", ascending=False)

print("\nStability distribution:")
for t in [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    print(f"  >= {t:.0%}: {(stability_fraction_ea >= t).sum()} genes")

results_df_ea.to_csv(os.path.join(ea_dir, "gene_doubleml_stability_smoking_EA.csv"), index=False)
print("\nSaved.")

EA confounders: (1460, 12)
Completed 5/30, elapsed 8.7s
Completed 10/30, elapsed 17.7s
Completed 15/30, elapsed 28.7s
Completed 20/30, elapsed 39.4s
Completed 25/30, elapsed 49.7s
Completed 30/30, elapsed 58.7s
Total time: 58.7s

Stability distribution:
  >= 50%: 541 genes
  >= 60%: 420 genes
  >= 70%: 330 genes
  >= 80%: 238 genes
  >= 90%: 157 genes
  >= 100%: 62 genes

Saved.


In [5]:
import pandas as pd
import os

aa_dir = r"C:\Users\user\Downloads\GSE148375_clean"
ea_dir = r"C:\Users\user\Downloads\GSE148812_clean"

# AA's 65-gene shortlist (100% stability, TAS1R3 duplicate removed)
results_df_aa = pd.read_csv(os.path.join(aa_dir, "gene_doubleml_stability_smoking_AA.csv"))
shortlist_100_aa = results_df_aa[results_df_aa["stability_fraction"] == 1.0]
aa_genes_100 = set(shortlist_100_aa["gene"].tolist()) - {"TAS1R3"}

# EA's 62-gene shortlist (100% stability)
results_df_ea = pd.read_csv(os.path.join(ea_dir, "gene_doubleml_stability_smoking_EA.csv"))
ea_genes_100 = set(results_df_ea[results_df_ea["stability_fraction"] == 1.0]["gene"].tolist())

print("AA genes at 100%:", len(aa_genes_100))
print("EA genes at 100%:", len(ea_genes_100))

overlap_pre_pc = aa_genes_100 & ea_genes_100
print(f"\nOverlap BEFORE PC algorithm: {len(overlap_pre_pc)} genes")
print(sorted(overlap_pre_pc))

AA genes at 100%: 65
EA genes at 100%: 62

Overlap BEFORE PC algorithm: 4 genes
['CP', 'DNAH1', 'FRAS1', 'HYDIN']


In [6]:
from scipy.stats import hypergeom

M = 15309  # total protein-coding genes tested in AA (population size)
n = 65     # AA shortlist size
N = 62     # EA shortlist size
k = 4      # observed overlap

p_value = hypergeom.sf(k-1, M, n, N)
print(f"Probability of >= {k} overlapping genes by chance: {p_value:.6f}")

Probability of >= 4 overlapping genes by chance: 0.000137


In [7]:
aa_pc_shortlist_33 = ['ZNF805', 'DVL1', 'LAMA5', 'DNAH5', 'DCHS2', 'USH2A', 'TICAM1', 'CELSR2', 
                      'PLEC', 'RYR1', 'FRAS1', 'XIRP1', 'CHD6', 'RNF213', 'ZNF418', 'FREM1',
                      'FREM2', 'DNAH1', 'GPR98', 'ASTN1', 'HYDIN', 'SIGLEC1', 'EXO1', 'LZTFL1', 
                      'TSC2', 'HMCN1', 'RTN4', 'UTP20', 'CNTRL', 'CCBL2', 'MDN1', 'COL6A6', 'CPAMD8']

overlap_genes = ['CP', 'DNAH1', 'FRAS1', 'HYDIN']
aa_pc_survivors = [g for g in overlap_genes if g in aa_pc_shortlist_33]
print("Of the 4 pre-PC overlap genes, these survived AA's PC algorithm:", aa_pc_survivors)

Of the 4 pre-PC overlap genes, these survived AA's PC algorithm: ['DNAH1', 'FRAS1', 'HYDIN']
